# Performance Analysis — Mean Reversion (Multi-Symbol)

Read the CSV file by running `python scripts/export_history.py` from the repo root.
Default file: `data/results/trades.csv` (legacy fallback: `data/results/trades_eurusd.csv`).
The notebook skips gracefully when no data exists and validates `time`, `profit` columns before plotting.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
CANDIDATES = [
    ROOT / "data" / "results" / "trades.csv",
    ROOT / "data" / "results" / "trades_eurusd.csv",
]
csv_path = next((p for p in CANDIDATES if p.exists()), None)
if csv_path is None:
    print("No data yet. Run: python scripts/export_history.py")
    print("Checked:")
    [print(f" - {p}") for p in CANDIDATES]
    df = pd.DataFrame()
else:
    print(f"Reading: {csv_path}")
    df = pd.read_csv(csv_path)

if not df.empty:
    missing = {"time", "profit"} - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}. Found: {list(df.columns)}")
    if pd.api.types.is_numeric_dtype(df["time"]):
        df["time"] = pd.to_datetime(df["time"], unit="s")
    else:
        df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time").reset_index(drop=True)
    df["cum_pnl"] = df["profit"].cumsum()
    df["drawdown"] = df["cum_pnl"] - df["cum_pnl"].cummax()

if df.empty:
    print("No deals to analyze.")
else:
    total = df["profit"].sum()
    n = len(df)
    win_rate = (df["profit"] > 0).mean()
    gross_win = df.loc[df["profit"] > 0, "profit"].sum()
    gross_loss = df.loc[df["profit"] < 0, "profit"].abs().sum()
    pf = gross_win / gross_loss if gross_loss != 0 else float("inf") if gross_win > 0 else 0.0
    max_dd = df["drawdown"].min()
    print(f"Total profit: ${total:.2f}")
    print(f"Trades: {n}")
    print(f"Win rate: {win_rate:.1%}")
    print(f"Profit Factor: {pf:.2f}")
    print(f"Max Drawdown: ${max_dd:.2f}")
    if "symbol" in df.columns:
        print("\nBy symbol:")
        print(df.groupby("symbol")["profit"].agg(count="size", pnl="sum"))

if not df.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    ax1.plot(df["time"], df["cum_pnl"], color="green")
    ax1.set_title("Equity Curve - Mean Reversion (Multi-Symbol)")
    ax1.set_ylabel("Cumulative PnL (USD)")
    ax1.grid(True)
    ax2.fill_between(df["time"], df["drawdown"], 0, color="red", alpha=0.3)
    ax2.set_ylabel("Drawdown (USD)")
    ax2.grid(True)
    plt.tight_layout()
    plt.show()